<a href="https://colab.research.google.com/github/abhisinghh/RAG_over_doc/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ragdoc demo: agentic RAG over a PDF

This notebook is a thin client: all logic lives in the `ragdoc` package. It installs the package, indexes a PDF, asks questions with citations, compares the baseline with the agentic loop, and launches the Gradio UI.

In [ ]:
%pip install -q "ragdoc[ui] @ git+https://github.com/abhisinghh/RAG_over_doc.git"

In [ ]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

In [ ]:
# Upload a PDF (Colab) or set PDF_PATH to a local file.
PDF_PATH = None
try:
    from google.colab import files
    PDF_PATH = next(iter(files.upload()))
except ImportError:
    PDF_PATH = PDF_PATH or "your-document.pdf"
PDF_PATH

In [ ]:
from ragdoc import load_settings
from ragdoc.engine import RagEngine

engine = RagEngine(load_settings())
report = engine.ingest(PDF_PATH)
report

In [ ]:
result = engine.ask("What is this document about, and what are its main conclusions?")
print(result.answer, "\n")
for c in result.citations:
    print(f"[{c.marker}] {c.source} p.{c.pages}: {c.snippet[:120]}")
print("\nroute:", result.route, "| steps:", [e.step for e in result.trace], "| tokens:", result.usage)

## Baseline vs agentic

The same index, answered by the single-pass dense baseline (close to the original repo) and by the full agentic loop.

In [ ]:
# Share the models and the vector store (embedded Qdrant allows one client per folder).
baseline = RagEngine(
    load_settings(overrides={"retrieval": {"mode": "dense", "reranker": "none"}, "agent": {"enabled": False}}),
    llm=engine.llm, embedder=engine.embedder, vector_store=engine.vectors,
)

q = "Compare the two most important findings and explain how they relate."
for name, eng in [("baseline", baseline), ("agentic", engine)]:
    r = eng.ask(q, use_cache=False)
    print(f"--- {name} ({r.latency_ms:.0f} ms, {r.usage.calls} calls)\n{r.answer}\n")

In [ ]:
from ragdoc.ui.gradio_app import build_ui

build_ui(engine).launch(share=True)